# NTK Results

## Preliminaries

In [ ]:
from jax import config
config.update("jax_enable_x64", True)

In [ ]:
import pickle
import os

import jax
import jax.numpy as jnp

from src.utils import *
from src.functions import *
from src.pdes import *
from src.ntk import *

from flax import nnx
import optax

from jaxkan.KAN import KAN

from sklearn.model_selection import train_test_split

## Function Fitting

We first perform experiments relevant to the NTK for the Function Fitting case, because PDEs have their own NTK formulation.

### Parameters

In [ ]:
N = 5000
n_ntk = 256

seed = 42

num_epochs = 2001
checkpoints = [0, 500, 1000, 1500, 2000]

opt_type = optax.adam(learning_rate=0.001)

pows = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0]

# Model input/output
n_in, n_out = 2, 1

# Studied functions
funcs = [("f3", f3)]

In [ ]:
# --------------------------
# Small architecture details
# --------------------------
G_small = 5
hidden_small = [8, 8]

# ------------------------
# Big architecture details
# ------------------------
G_big = 20
hidden_big = [32, 32, 32]

### Helper Functions

In [ ]:
def get_data(func, N, n_ntk, seed):
    
    # Generate data
    x, y = generate_func_data(func, 2, N, seed)
    
    # Split data (at this point just to ensure continuity)
    X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=seed)
    
    # Subsample points used to compute NTK
    key_ntk = jax.random.PRNGKey(seed)
    idx = jax.random.choice(key_ntk, X_train.shape[0], shape=(n_ntk,), replace=False)
    X_ntk = X_train[idx]

    return X_train, y_train, X_ntk

In [ ]:
def run_experiment(func, model, opt, X_train, y_train, X_ntk, title):
    spec_list, tau_list = [], []
    conds, ranks = [], []

    # τ = 0 (before any updates)
    K0 = stabilize_kernel(ntk_matrix(model, X_ntk))
    lam0 = jnp.sort(jnp.linalg.eigvalsh(K0))[::-1]
    spec_list.append(lam0)
    tau_list.append(0)
    conds.append(cond_from_eigs(lam0))
    
    eff_rank0 = (lam0.sum() ** 2) / (jnp.sum(lam0 ** 2) + 1e-12)
    ranks.append(float(eff_rank0))

    for epoch in range(num_epochs):
        loss = func_fit_step(model, opt, X_train, y_train)

        if epoch in checkpoints[1:]:
            Kt = stabilize_kernel(ntk_matrix(model, X_ntk))
            lam = jnp.sort(jnp.linalg.eigvalsh(Kt))[::-1]
            spec_list.append(lam)
            tau_list.append(epoch)
            conds.append(cond_from_eigs(lam))
            
            eff_rank_t = (lam.sum() ** 2) / (jnp.sum(lam ** 2) + 1e-12)
            ranks.append(float(eff_rank_t))

    l2error = func_fit_eval(model, func, 2, 200)

    print(f"\t{title} Model Metrics:")
    print(f"\tCond Number: τ=0 → {conds[0]:.2e}, τ={tau_list[-1]} → {conds[-1]:.2e}")
    print(f"\tEffective Rank: τ=0 → {ranks[0]:.2f}, τ={tau_list[-1]} → {ranks[-1]:.2f}")
    print(f"\tFinal Loss = {loss:.2e}\t L^2 Error = {l2error:.2e}\n")

    return spec_list, tau_list, conds, ranks

### Main Routine

In [ ]:
results = dict()

for func_name, func in funcs:
    
    results[func_name] = dict()

    for pow_basis in pows:
    
        results[func_name][pow_basis] = dict()

        for pow_res in pows:
    
            results[func_name][pow_basis][pow_res] = dict()

            params_small_power = {'k': 3, 'G': G_small, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                      'init_scheme': {'type': 'power', "const_b": 1.0, "const_r": 1.0, "pow_b1": pow_basis, "pow_b2": pow_basis, "pow_r1": pow_res, "pow_r2": pow_res}}

            params_big_power = {'k': 3, 'G': G_big, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                      'init_scheme': {'type': 'power', "const_b": 1.0, "const_r": 1.0, "pow_b1": pow_basis, "pow_b2": pow_basis, "pow_r1": pow_res, "pow_r2": pow_res}}

            # Get the data for the function
            X_train, y_train, X_ntk = get_data(func, N, n_ntk, seed)
        
            # Define the small architecture
            layer_dims = [n_in, *hidden_small, n_out]
        
            results[func_name][pow_basis][pow_res]["small"] = dict()
            
            print(f"Training model with dimensions {layer_dims} for function {func_name}.")
        
            # Power Law
            results[func_name][pow_basis][pow_res]["small"]["Power"] = dict()
            
            power_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_small_power, seed = seed+1)
            power_opt = nnx.Optimizer(power_model, opt_type)
        
            spec_list, tau_list, conds, ranks = run_experiment(func, power_model, power_opt, X_train, y_train, X_ntk, "Power")
            results[func_name][pow_basis][pow_res]["small"]["Power"]["spec_list"] = spec_list
            results[func_name][pow_basis][pow_res]["small"]["Power"]["tau_list"] = tau_list
            results[func_name][pow_basis][pow_res]["small"]["Power"]["conds"] = conds
            results[func_name][pow_basis][pow_res]["small"]["Power"]["ranks"] = ranks
        
        
            # Define the big architecture
            layer_dims = [n_in, *hidden_big, n_out]
        
            results[func_name][pow_basis][pow_res]["big"] = dict()
        
            print(f"Training model with dimensions {layer_dims} for function {func_name}.")
        
            # Power Law
            results[func_name][pow_basis][pow_res]["big"]["Power"] = dict()
            
            power_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_big_power, seed = seed+1)
            power_opt = nnx.Optimizer(power_model, opt_type)
        
            spec_list, tau_list, conds, ranks = run_experiment(func, power_model, power_opt, X_train, y_train, X_ntk, "Power")
            results[func_name][pow_basis][pow_res]["big"]["Power"]["spec_list"] = spec_list
            results[func_name][pow_basis][pow_res]["big"]["Power"]["tau_list"] = tau_list
            results[func_name][pow_basis][pow_res]["big"]["Power"]["conds"] = conds
            results[func_name][pow_basis][pow_res]["big"]["Power"]["ranks"] = ranks

In [ ]:
# Save results for further processing
results_dir = 'ff_results/'

with open(os.path.join(results_dir, "ntk_powgrid.pkl"), "wb") as f:
    pickle.dump(results, f)